# Análise Numérica de Sistemas Lineares com a Matriz de Hilbert: Eliminação Gaussiana e Cálculo do Determinante

**Disciplina**: Cálculo Numérico  
**Professor**: Marcos Maia  
**Autor**: Yann Keven Jordão Leão (Engenharia da Computação - UFRPE)  

## Introdução

Este notebook tem como objetivo resolver um sistema linear da forma **H·x = b**, onde **H** é uma matriz de Hilbert, dada por:

$$
H_{ij} = \frac{1}{i+j-1}
$$

e **b** é um vetor construído como a soma das linhas da matriz H, logo:

$$
b_i = \sum_{j=1}^{n} H_{ij}
$$

A matriz de Hilbert é conhecida por ser uma matriz **simétrica, positiva definida**, mas **mal-condicionada** numericamente, o que a torna um excelente exemplo para testar a estabilidade de algoritmos de resolução de sistemas lineares. 

In [ ]:
# Definição da Matriz de Hilbert
def matriz_hilbert(dimensao):
    H = H = [[0.0 for _ in range(dimensao)] for _ in range(dimensao)]
    
    for i in range(dimensao):
        for j in range(dimensao):
            H[i][j] = 1 / (i + j + 1)

    return H

Note que, como em Python os ídices `i` e `j` começam em `0`, para que bata com a fórmula matemática (que começa do 1), precisamos somar 1 a ambos os índices:

$$
H_{ij} = \frac{1}{(i+1) + (j+1) - 1} = \frac{1}{i + j + 1}
$$

In [ ]:
# Vetor dos coeficientes independentes
def vetor_independente(matriz):
    return [sum(linha) for linha in matriz]

## Generalização do vetor solução

Note que, para qualquer dimensão **n** da Matriz **H** o vetor solução terá a forma:

$$
\begin{pmatrix}
1 & 1 & 1 & ... & 1
\end{pmatrix}
$$

Essa generalização ocorre devido à definição do vetor **b**, cujos elementos são a soma dos coeficientes de cada linha da matriz H.
Como resultado, ao multiplicar H pelo vetor solução composto apenas por 1s, cada linha da matriz é somada, reproduzindo exatamente os valores de b. Isso garante que o vetor com todos os elementos iguais a 1 seja, de fato, uma solução do sistema linear $H·x = b$.

## Eliminação Gaussiana para Diferentes Dimensões de $H$

Implementaremos o algoritmo da Eliminação Gaussiana sem pivotamento para resolver sistemas lineares da forma $H·x = b$.
Vamos testar a implementação para diferentes tamanhos de matriz.

In [ ]:
# Eliminação Gaussiana (sem pivoteamento)
def eliminacao_gauss(matriz, vetor):
    
    # Copiando os parâmetros para evitar efeitos colaterais
    from copy import deepcopy
    dimensao = len(matriz)
    H = deepcopy(matriz)
    b = vetor[:]
    
    for k in range(dimensao - 1):
        for i in range(k + 1, dimensao):
            mult = H[i][k] / H[k][k]
            for j in range(k, dimensao):
                H[i][j] -= mult * H[k][j]
            b[i] -= mult * b[k]       
            
    return H, b


In [ ]:
# Substituição Reversa
def substituicao_reversa(matriz_escalonada, vetor):
    dimensao = len(matriz_escalonada) 
    resultado = [0.0 for _ in range(dimensao)]
    resultado[dimensao - 1] = vetor[dimensao - 1] / matriz_escalonada[dimensao - 1][dimensao - 1]

    for k in range(dimensao - 1, -1, -1):
        soma = 0
        for j in range(k + 1, dimensao):
            soma += matriz_escalonada[k][j] * resultado[j]

        resultado[k] = (vetor[k] - soma) / matriz_escalonada[k][k]

    return resultado

In [ ]:
# Geração da matrizes
H_3 = matriz_hilbert(3)
H_10 = matriz_hilbert(10)
H_100 = matriz_hilbert(100)

# Geração dos vetores
b_3 = vetor_independente(H_3)
b_10 = vetor_independente(H_10)
b_100 = vetor_independente(H_100)

- $n = 3$

In [ ]:
# Teste com n = 3
Ht_3, bt_3 = eliminacao_gauss(H_3, b_3)

resultado_3 = substituicao_reversa(Ht_3, bt_3)

In [ ]:
print(f'Resultados da matriz de dimensão 3: {resultado_3}')

- $n = 10$


In [ ]:
# Teste com n = 10
Ht_10, bt_10 = eliminacao_gauss(H_10, b_10)

resultado_10 = substituicao_reversa(Ht_10, bt_10)

In [ ]:
print('Resultados da matriz de dimensão 10:')
for valor in resultado_10:
    print(valor)

- $n = 100$

In [ ]:
# Teste com n = 100
Ht_100, bt_100 = eliminacao_gauss(H_100, b_100)

resultado_100 = substituicao_reversa(Ht_100, bt_100)

In [ ]:
print('Resultados da matriz de dimensão 100:')
for valor in resultado_100:
    print(valor)

### O que os resultados nos dizem?

Ao resolver o sistema com a matriz de Hilbert de dimensão 100, observamos que os valores obtidos se desviam fortemente do esperado — como explicado, um vetor como $\begin{pmatrix}1 & 1 & 1 & \dots & 1\end{pmatrix}$. Em vez disso, surgem valores como `308` e `-272`, o que indica um resultado numericamente instável.

Isso ocorre porque a matriz de Hilbert é notoriamente mal condicionada para dimensões maiores: suas entradas são **muito próximas de zero**, o que leva a um sistema quase singular. Pequenas imprecisões numéricas no processo de eliminação de Gauss são amplificadas, produzindo soluções distantes da realidade esperada.

## Cálculo Numérico do Determinante a partir da Matriz Escalonada

Nesta etapa, calcularemos numericamente o **determinante** da matriz dos coeficientes $H$ para os casos:

- $n = 3$
- $n = 10$
- $n = 100$

O cálculo será feito com base na matriz escalonada obtida durante o processo de anterior. Como o método transforma a matriz original em uma forma triangular superior, o determinante pode ser obtido simplesmente como o **produto dos elementos da diagonal principal** da matriz escalonada:

$$
\det(H) = \prod_{i=1}^n H_{ii}
$$




In [ ]:
# Função para o determinante das matrizes escalonadas
def determinante_triangular(matriz):
    determinante = 1
    for i in range(len(matriz)):
        determinante *= matriz[i][i]
    return determinante

In [ ]:
# Cálculo dos determinantes
det_3 = determinante_triangular(Ht_3)
det_10 = determinante_triangular(Ht_10)
det_100 = determinante_triangular(Ht_100)

In [ ]:
print(f"""
Determinantes das Matrizes de Hilbert:

Dimensão   3   ⇒  det = {det_3}
Dimensão  10   ⇒  det = {det_10}
Dimensão 100   ⇒  det = {det_100}
""")

### Valor dos determinantes

A análise dos determinantes evidencia a crescente instabilidade numérica conforme a dimensão da matriz de Hilbert aumenta. Quando $n = 100$, o determinante calculado é tão pequeno que pode ser interpretado como zero pela máquina, devido ao limite de precisão numérica. Isso, em teoria, indicaria um sistema singular — ou seja, sem solução única.

No entanto, como discutido no início do notebook, o sistema foi construído de forma que a solução exata seja $x_1 = x_2 = \dots = x_{100} = 1$. O que ocorre é que, apesar da matriz ser quase singular, o método ainda retorna uma solução próxima da correta — mas com grandes erros em algumas entradas, como valores absurdamente altos ou negativos.

Já para $n = 10$, o determinante, embora pequeno $10^{-53}$, ainda permite uma solução relativamente estável. Isso se reflete nos resultados obtidos, com pequenas variações em torno de 1, como `0.9994...`, o que é aceitável dentro dos limites da precisão numérica.

## Conclusão

Matrizes de Hilbert, embora teoricamente bem definidas, tornam-se numericamente instáveis à medida que sua dimensão aumenta. Isso se reflete em determinantes extremamente pequenos e em soluções com grandes erros devido à perda de precisão. Assim, são ótimos exemplos para ilustrar os limites da aritmética de ponto flutuante e a importância do condicionamento em sistemas lineares.